<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Borexino Canonical Data Generator

Downloads the official Nature 2018 open tables and exports detector observations and experimental survival-probability points. These are not solar production spectra.

## 1. Imports and paths

In [1]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Download and convert

In [2]:
PROVIDER_DIR = SOLAR_DATA / "borexino"
for category in ("raw", "probability", "observation", "metadata"):
    (PROVIDER_DIR / category).mkdir(parents=True, exist_ok=True)
URLS = {"Nature2018_Fig2a_DATA.txt": "https://borex.lngs.infn.it/wp-content/uploads/OpenData/Nature2018_Fig2a_DATA.txt", "Nature2018_Fig2b_DATA.txt": "https://borex.lngs.infn.it/wp-content/uploads/OpenData/Nature2018_Fig2b_DATA.txt", "Nature2018_Fig3_DATApoints.txt": "https://borex.lngs.infn.it/wp-content/uploads/OpenData/Nature2018_Fig3_DATApoints.txt"}
paths = {name: download(url, PROVIDER_DIR / "raw" / name) for name, url in URLS.items()}

def numeric_rows(path, ncols):
    rows = []
    for line in path.read_text(errors="replace").splitlines():
        tokens = line.split()
        if len(tokens) != ncols:
            continue
        try: rows.append([float(token) for token in tokens])
        except ValueError: pass
    return rows

low = pd.DataFrame(numeric_rows(paths["Nature2018_Fig2a_DATA.txt"], 7), columns=["bin", "N_h", "energy_keV", "width_keV", "rate", "rate_error", "residual"])
low.to_csv(PROVIDER_DIR / "observation" / "nature2018_low_energy_spectrum.csv", index=False)
high = pd.DataFrame(numeric_rows(paths["Nature2018_Fig2b_DATA.txt"], 5), columns=["bin", "radius_m", "counts", "error", "residual"])
high.to_csv(PROVIDER_DIR / "observation" / "nature2018_high_energy_radial.csv", index=False)

lines = paths["Nature2018_Fig3_DATApoints.txt"].read_text(errors="replace").splitlines()
probability_rows, source = [], None
for line in lines:
    stripped = line.strip()
    if stripped in {"pp", "Be7", "pep", "B8"}:
        source = {"Be7": "7Be", "B8": "8B"}.get(stripped, stripped)
        continue
    tokens = stripped.split()
    if source and len(tokens) == 4:
        try:
            energy, pee, up, down = map(float, tokens)
        except ValueError:
            continue
        probability_rows.append([source, energy, pee, down, up])
        source = None
probability = pd.DataFrame(probability_rows, columns=["source", "energy_MeV", "probability", "uncertainty_minus", "uncertainty_plus"])
probability.to_csv(PROVIDER_DIR / "probability" / "nature2018_pee_points.csv", index=False)
(PROVIDER_DIR / "metadata" / "source.json").write_text(json.dumps({"provider": "Borexino", "doi": "10.1038/s41586-018-0624-y", "url": "https://borex.lngs.infn.it/papers/articles/solar_nu/comprehensive-measurement-of-pp-chain-solar-neutrinos/", "probability_kind": "experimental points"}, indent=2) + "\n")
display(low.head(), high.head(), probability)

,bin,N_h,energy_keV,width_keV,rate,rate_error,residual
0,1.0,92.0,212.185,2.35242,19.70540,0.183954,19.70540
1,2.0,93.0,214.537,2.35351,16.42550,0.167948,-7.63806
2,3.0,94.0,216.891,2.35459,11.73910,0.141982,-2.67957
3,4.0,95.0,219.245,2.35568,9.34528,0.126681,-4.48962
4,5.0,96.0,221.601,2.35676,6.94799,0.109231,2.53581


,bin,radius_m,counts,error,residual
0,1.0,0.05,0.0,0.0,0.0
1,2.0,0.15,0.0,0.0,0.0
2,3.0,0.25,0.0,0.0,0.0
3,4.0,0.35,0.0,0.0,0.0
4,5.0,0.45,0.0,0.0,0.0


,source,energy_MeV,probability,uncertainty_minus,uncertainty_plus
0,pp,0.267,0.566,0.092,0.092
1,7Be,0.862,0.532,0.054,0.054
2,pep,1.440,0.428,0.114,0.114
3,8B,7.400,0.390,0.090,0.090


## 3. Validation

In [3]:
assert len(low) > 0 and len(high) > 0
assert set(probability["source"]) == {"pp", "7Be", "pep", "8B"}
assert probability["probability"].between(0, 1).all()
print("Borexino open-data products validated.")

Borexino open-data products validated.
